In [113]:
import pandas as pd
import numpy as np

tkpi = pd.read_csv("tkpi_with_category.csv")
urt = pd.read_csv("urt_with_category.csv")



In [114]:
tkpi.columns

Index(['food_code', 'food_name', 'source', 'water_g_100g', 'energy_kcal_100g',
       'protein_g_100g', 'fat_g_100g', 'carb_g_100g', 'fiber_g_100g',
       'ash_g_100g', 'calcium_mg_100g', 'phosphorus_mg_100g', 'iron_mg_100g',
       'sodium_mg_100g', 'potassium_mg_100g', 'copper_mg_100g', 'zinc_mg_100g',
       'retinol_mcg_100g', 'beta_carotene_mcg_100g', 'carotene_total_mcg_100g',
       'thiamin_mg_100g', 'riboflavin_mg_100g', 'niacin_mg_100g',
       'vitamin_c_mg_100g', 'bdd_percent', 'tkpi_main_group',
       'tkpi_group_code', 'category_code', 'category_name', 'category_source',
       'food_name_clean', 'food_name_match'],
      dtype='object')

In [115]:
urt.columns


Index(['food_name', 'weight_gram', 'urt', 'category_code', 'category_name',
       'urt_qty', 'urt_unit', 'urt_food_name_clean', 'urt_food_name_match'],
      dtype='object')

In [116]:
import pandas as pd

# ============================================================
# 1. Prepare merge keys
# ============================================================

tkpi_merge = tkpi.copy()
urt_merge = urt.copy()

tkpi_merge["category_code"] = (
    tkpi_merge["category_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

urt_merge["category_code"] = (
    urt_merge["category_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

tkpi_merge["food_name_match"] = (
    tkpi_merge["food_name_match"]
    .astype(str)
    .str.strip()
    .str.lower()
)

urt_merge["urt_food_name_match"] = (
    urt_merge["urt_food_name_match"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# ============================================================
# 1B. Manual mapping BEFORE merge
# ============================================================

manual_mapping = {
    "ayam, daging": "daging ayam",

    "sapi, daging, gemuk": "daging sapi",
    "sapi, daging, kurus": "daging sapi",
    "sapi, daging, lemak sedang": "daging sapi",

    "anak sapi, daging, gemuk": "daging sapi",
    "anak sapi, daging, kurus": "daging sapi",
    "anak sapi, daging, sedang": "daging sapi",

   "beras giling": "nasi beras giling putih",
    "beras giling var pelita": "nasi beras giling putih",
    "beras giling var rojolele": "nasi beras giling putih",

    "beras hitam": "nasi beras giling hitam",
    "beras tumbuk merah": "nasi beras giling merah",

}

tkpi_merge["food_name_match_original"] = tkpi_merge["food_name_match"]

tkpi_merge["food_name_match"] = (
    tkpi_merge["food_name_match"]
    .replace(manual_mapping)
)

# ============================================================
# 2. Prepare URT columns for merge
# ============================================================

urt_for_merge = urt_merge.rename(
    columns={
        "urt_food_name_match": "food_name_match",
        "weight_gram": "gram_per_portion"
    }
)

urt_for_merge = urt_for_merge[
    [
        "category_code",
        "food_name_match",
        "urt",
        "urt_qty",
        "urt_unit",
        "gram_per_portion"
    ]
].copy()

urt_for_merge = urt_for_merge.drop_duplicates(
    subset=["category_code", "food_name_match"],
    keep="first"
)

# ============================================================
# 3. Merge TKPI + URT
# ============================================================

merged = tkpi_merge.merge(
    urt_for_merge,
    on=["category_code", "food_name_match"],
    how="left"
)

# ============================================================
# 4. Check result
# ============================================================

print("Total TKPI rows:", len(tkpi_merge))
print("Total merged rows:", len(merged))

print("\nMerged preview:")
print(
    merged[
        [
            "food_code",
            "food_name",
            "category_code",
            "food_name_match_original",
            "food_name_match",
            "urt",
            "gram_per_portion"
        ]
    ].head(30)
)

# Check manual mapping result
manual_mapped_rows = merged[
    merged["food_name_match_original"] != merged["food_name_match"]
]



Total TKPI rows: 1145
Total merged rows: 1145

Merged preview:
   food_code                            food_name category_code  \
0      AR001                 Beras giling, mentah            MP   
1      AR002      Beras giling var pelita, mentah            MP   
2      AR003    Beras giling var rojolele, mentah            MP   
3      AR004                  Beras hitam, mentah            MP   
4      AR005  Beras jagung kuning, kering, mentah            MP   
5      AR006   Beras jagung putih, kering, mentah            MP   
6      AR007     Beras ketan hitam tumbuk, mentah            MP   
7      AR008     Beras ketan putih tumbuk, mentah            MP   
8      AR009                 Beras ladang, mentah            MP   
9      AR010                  Beras menir, mentah            MP   
10     AR011                      Beras parboiled            MP   
11     AR012                 Beras tumbuk, mentah            MP   
12     AR013           Beras tumbuk merah, mentah            MP   

In [117]:
merged.category_code.value_counts()

category_code
LH     310
MP     241
S      216
LN     135
B      121
NAN     52
G       34
M       19
SS      17
Name: count, dtype: int64

In [118]:
merged.columns

Index(['food_code', 'food_name', 'source', 'water_g_100g', 'energy_kcal_100g',
       'protein_g_100g', 'fat_g_100g', 'carb_g_100g', 'fiber_g_100g',
       'ash_g_100g', 'calcium_mg_100g', 'phosphorus_mg_100g', 'iron_mg_100g',
       'sodium_mg_100g', 'potassium_mg_100g', 'copper_mg_100g', 'zinc_mg_100g',
       'retinol_mcg_100g', 'beta_carotene_mcg_100g', 'carotene_total_mcg_100g',
       'thiamin_mg_100g', 'riboflavin_mg_100g', 'niacin_mg_100g',
       'vitamin_c_mg_100g', 'bdd_percent', 'tkpi_main_group',
       'tkpi_group_code', 'category_code', 'category_name', 'category_source',
       'food_name_clean', 'food_name_match', 'food_name_match_original', 'urt',
       'urt_qty', 'urt_unit', 'gram_per_portion'],
      dtype='object')

Add 1 gelas and 100 grams per portion

In [119]:
# ============================================================
# Set standard portion for vegetables
# ============================================================

vegetable_mask = merged["category_code"].eq("S")

merged.loc[vegetable_mask, "urt"] = "1 gelas"
merged.loc[vegetable_mask, "gram_per_portion"] = 100
merged.loc[vegetable_mask, "urt_qty"] = 1.0
merged.loc[vegetable_mask, "urt_unit"] = "gelas"

In [120]:
output_file = "tkpi_urt_merged.csv"

merged.to_csv(output_file, index=False, encoding="utf-8-sig")

In [121]:
portion_by_category = (
    merged
    .assign(
        portion_available=merged["urt"].notna() & merged["gram_per_portion"].notna()
    )
    .groupby("category_code")["portion_available"]
    .agg(
        total="count",
        available="sum"
    )
    .reset_index()
)

portion_by_category["missing"] = (
    portion_by_category["total"] - portion_by_category["available"]
)

portion_by_category["available_percentage"] = (
    portion_by_category["available"] / portion_by_category["total"] * 100
).round(2)

print(portion_by_category)

  category_code  total  available  missing  available_percentage
0             B    121         22       99                 18.18
1             G     34          3       31                  8.82
2            LH    310         14      296                  4.52
3            LN    135          8      127                  5.93
4             M     19          5       14                 26.32
5            MP    241         13      228                  5.39
6           NAN     52          0       52                  0.00
7             S    216        216        0                100.00
8            SS     17          3       14                 17.65


In [122]:

portion_available_df = merged[
    merged["urt"].notna() &
    merged["gram_per_portion"].notna()
].copy()

print("Total foods in merged data:", len(merged))
print("Total foods with available portion:", len(portion_available_df))
print("Percentage available:", round(len(portion_available_df) / len(merged) * 100, 2), "%")

Total foods in merged data: 1145
Total foods with available portion: 284
Percentage available: 24.8 %


In [123]:
# ============================================================
# Count portion-available foods by category
# ============================================================

portion_available_by_category = (
    portion_available_df
    .groupby(["category_code", "category_name"])
    .size()
    .reset_index(name="available_count")
    .sort_values("category_code")
)

print(portion_available_by_category)

  category_code   category_name  available_count
0             B            Buah               22
1             G            Gula                3
2            LH     Lauk Hewani               14
3            LN     Lauk Nabati                8
4             M  Minyak / Lemak                5
5            MP   Makanan Pokok               13
6             S           Sayur              216
7            SS            Susu                3


In [124]:
# 1. Mark TKPI processed code
merged["is_processed_code"] = (
    merged["tkpi_group_code"]
    .astype(str)
    .str.upper()
    .str.endswith("P")
)

# 2. Build whitelist from URT
urt_whitelist = set(
    urt["urt_food_name_match"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

merged["exists_in_urt"] = (
    merged["food_name_match"]
    .astype(str)
    .str.lower()
    .str.strip()
    .isin(urt_whitelist)
)

# 3. Manual whitelist
manual_whitelist = [
    "nasi", "bihun", "makaroni", "mi basah", "mi kering",
    "roti", "tepung", "tahu", "oncom", "tempe",
    "susu kedelai", "sari kedelai",
]

merged["exists_in_manual_whitelist"] = merged["food_name_match"].apply(
    lambda x: any(w in str(x).lower() for w in manual_whitelist)
)

# 4. Remove rule
merged["remove_processed"] = (
    (merged["is_processed_code"] == True) &
    (merged["exists_in_urt"] == False) &
    (merged["exists_in_manual_whitelist"] == False)
)

# 5. Filter data
filtered = merged[merged["remove_processed"] == False].copy()


In [125]:
print("Before filtering:", len(merged))
print("After filtering:", len(filtered))
print("Removed processed:", merged["remove_processed"].sum())


Before filtering: 1145
After filtering: 645
Removed processed: 500


In [126]:
filtered.category_code.value_counts()

category_code
S      157
LH     133
B      108
LN      91
MP      85
NAN     27
G       21
M       17
SS       6
Name: count, dtype: int64

In [127]:
filtered_portion_ready = filtered[
    filtered["urt"].notna() &
    filtered["gram_per_portion"].notna()
].copy()

filtered_portion_ready.to_csv(
    "filtered_portion_ready_foods.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Portion-ready filtered data:", len(filtered_portion_ready))

Portion-ready filtered data: 225


In [128]:
filtered_portion_ready.category_code.value_counts()

category_code
S     157
B      22
LH     14
MP     13
LN      8
M       5
SS      3
G       3
Name: count, dtype: int64

In [129]:
filtered.to_csv(
    "filtered_food_data_selected_columns.csv",
    index=False,
    encoding="utf-8-sig"
)
